# ML-00 ML Input Export from 03 Feature Output

`data/preprocessing/03_ml_feature_process.ipynb`가 만든 `ml_exp00.parquet` 단일 산출물을 사용해 `ml_*.py` 모듈에 넣을 수 있는 입력 파일 세트를 생성합니다.

이 노트북은 **학습/검증/test를 실행하지 않습니다.** `fb_*.py`와 `ml_io.py`의 입력 검증 함수까지만 사용해 입력 산출물 생성 단계에서 멈춥니다.

생성 목표:

1. `ml_exp00.parquet` 단일 파일의 schema와 `split` 컬럼 확인
2. `ml_feature_columns.csv`의 `used_in_ml="TRUE"` feature 목록 확인
3. 단일 parquet를 `split` 컬럼 기준으로 train/val/test parquet 3개로 분리
4. `ml_feature_columns.csv`를 같은 output directory에 복사
5. 후속 `ml_train.py`, `ml_val.py`, `ml_test.py`에 넘길 경로를 manifest로 저장

## 실행 범위

포함:

- `fb_utils.py` 기준 프로젝트 경로 설정
- `fb_build.split_single_parquet_by_existing_split()` 실행
- ML 입력 파일 존재 여부와 schema 확인
- `ml_inputs_manifest.json` 저장

제외:

- `ml_train.py`, `ml_val.py`, `ml_test.py` import
- XGBoost 학습
- validation threshold 선택
- final test 평가
- metric 생성

## 실행 전 주의

- 원본 `data/processed/ml_features/ml_exp00.parquet`는 수정하지 않습니다.
- 원본 `ml_feature_columns.csv`는 수정하지 않고 output directory로 복사합니다.
- split parquet는 `ml/ml-00_baseline/outputs/feature_build/` 아래 새 run directory에 저장합니다.
- 기본 overwrite는 `False`입니다. 같은 설정으로 재실행하려면 `RUN_ID`를 바꾸는 것을 우선 권장합니다.
- 이 노트북이 만든 parquet/model/manifest 산출물은 commit 대상이 아닙니다.

# 00_경로 및 환경설정

In [ ]:
from __future__ import annotations
import json
import shutil
import sys
from pathlib import Path

import pandas as pd
# ============================================================
# 0.0 User Settings 
# ============================================================
# 실행 환경
IS_COLAB = False  # Colab 실행 시 True로 변경

# 입력 데이터 소스
# - "local": 코드 repo 내부 data/ 사용
# - "drive": Google Drive의 Graph_AML/data/ 사용
INPUT_DATA_SOURCE = "drive"

# 코드가 있는 로컬 Git repo root 경로
PROJECT_ROOT = Path("/mnt/d/AML_git/Graph_AML").resolve()

# Google Drive의 Graph_AML 경로
DRIVE_INPUT_PROJECT_ROOT_LOCAL = Path("/mnt/g/내 드라이브/Graph_AML").resolve()  # 로컬 WSL 기준
DRIVE_INPUT_PROJECT_ROOT_COLAB = Path("/content/drive/MyDrive/Graph_AML").resolve()  # Colab 기준

# 재현성 seed
SEED = 42

# ============================================================
# 0.1 Helpers
# ============================================================
def require_dir(path: Path, name: str, hint: str) -> Path:
    """필수 디렉터리 존재 여부를 시작 단계에서 명확히 검증한다."""
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}\n{hint}")
    if not path.is_dir():
        raise NotADirectoryError(f"{name} is not a directory: {path}")
    return path
# ============================================================
# 0.2 Runtime / Project Root Validation
# ============================================================
print(f"Kernel: {'Colab' if IS_COLAB else 'Local'}")
PROJECT_ROOT = require_dir(
    PROJECT_ROOT,
    "PROJECT_ROOT",
    "Set PROJECT_ROOT in 0.0 User Settings to your local Git repo path.",
)
# ============================================================
# 0.3 Input Data Root Selection
# ============================================================
if INPUT_DATA_SOURCE == "local":
    INPUT_PROJECT_ROOT = PROJECT_ROOT
elif INPUT_DATA_SOURCE == "drive":
    if IS_COLAB:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        INPUT_PROJECT_ROOT = DRIVE_INPUT_PROJECT_ROOT_COLAB
    else:
        INPUT_PROJECT_ROOT = DRIVE_INPUT_PROJECT_ROOT_LOCAL
else:
    raise ValueError(
        f'INPUT_DATA_SOURCE must be "local" or "drive", got: {INPUT_DATA_SOURCE}'
    )
INPUT_PROJECT_ROOT = require_dir(
    INPUT_PROJECT_ROOT,
    "INPUT_PROJECT_ROOT",
    "Check INPUT_DATA_SOURCE and Drive path values in 0.0 User Settings.",
)
# ============================================================
# 0.4 Module Directories
# ============================================================
# 코드 모듈은 항상 PROJECT_ROOT 기준으로 import한다.
MODULE_DIR = require_dir(
    PROJECT_ROOT / "ml" / "ml-00_baseline" / "feature_build",
    "MODULE_DIR",
    "PROJECT_ROOT must point to the local Graph_AML code repo root.",
)
ML_IO_DIR = require_dir(
    PROJECT_ROOT / "ml" / "ml-00_baseline" / "train_val_test",
    "ML_IO_DIR",
    "PROJECT_ROOT must point to the local Graph_AML code repo root.",
)
# ============================================================
# 0.5 Import Path Setup
# ============================================================
# ml_io는 train_val_test/ml_io.py를 우선 사용해야 하므로 ML_IO_DIR을 MODULE_DIR보다 앞에 둔다.
module_paths = [str(ML_IO_DIR), str(MODULE_DIR)]
sys.path = [path for path in sys.path if path not in module_paths]
sys.path[:0] = module_paths
# 노트북 셀 재실행 시 이전 import cache가 남을 수 있으므로 관련 모듈만 제거한다.
for module_name in ("fb_utils", "fb_build", "fb_io", "ml_io"):
    sys.modules.pop(module_name, None)
import fb_utils
from fb_build import split_single_parquet_by_existing_split
from fb_io import parquet_columns
from ml_io import (
    check_feature_columns_file,
    feature_columns_hash,
    normalize_feature_columns_file,
)
fb_utils.set_seed(SEED, use_torch=False)
# ============================================================
# 0.6 I/O Paths
# ============================================================
# Output: 항상 local code repo 기준
BASE_DIR = PROJECT_ROOT
PROCESSED_DIR = BASE_DIR / "data" / "processed"
# Input: local/drive 선택 결과 기준
INPUT_PROCESSED_DIR = INPUT_PROJECT_ROOT / "data" / "processed"
SOURCE_ML_FEATURE_DIR = require_dir(
    INPUT_PROCESSED_DIR / "ml_features",
    "SOURCE_ML_FEATURE_DIR",
    f"Check if ml_features exists for INPUT_DATA_SOURCE={INPUT_DATA_SOURCE}.",
)
# ============================================================
# 0.7 Configuration Summary
# ============================================================
print("\n" + "=" * 60)
print("Configuration Summary")
print("=" * 60)
print(f"Kernel              : {'Colab' if IS_COLAB else 'Local'}")
print(f"INPUT_DATA_SOURCE   : {INPUT_DATA_SOURCE}")
print(f"PROJECT_ROOT        : {PROJECT_ROOT}")
print(f"INPUT_PROJECT_ROOT  : {INPUT_PROJECT_ROOT}")
print(f"SOURCE_ML_FEATURE   : {SOURCE_ML_FEATURE_DIR}")
print(f"MODULE_DIR          : {MODULE_DIR}")
print(f"ML_IO_DIR           : {ML_IO_DIR}")
print(f"PROCESSED_DIR (OUT) : {PROCESSED_DIR}")
print(f"fb_utils.py         : {Path(fb_utils.__file__).resolve()}")
print("=" * 60 + "\n")

# 01_실행 설정

아래 셀만 수정해서 입력 산출물 위치, output directory, overwrite 정책을 제어합니다.

이 노트북은 full split parquet를 만드는 산출물 생성 단계입니다. sample 학습 옵션은 없습니다.

In [ ]:
# ------------------------------------------------------------
# 1.1 Experiment and run identifiers
# ------------------------------------------------------------
SOURCE_EXPERIMENT_ID = "ml_exp00"
EXPORT_EXPERIMENT_ID = "ml_exp00_test_run"
EXPORT_NAME = "ml-00-ml-input-export"
RUN_ID = "run_000"

# 정답 라벨 컬럼명
LABEL_COL = "label"


# ------------------------------------------------------------
# 1.2 Source input paths
# ------------------------------------------------------------
# SOURCE_ML_FEATURE_DIR는 bootstrap에서 이미 결정됨 (INPUT_PROJECT_ROOT 기준)
# - local: {PROJECT_ROOT}/data/processed/ml_features
# - drive: {DRIVE_INPUT_PROJECT_ROOT}/data/processed/ml_features
INPUT_SINGLE_PARQUET = SOURCE_ML_FEATURE_DIR / f"{SOURCE_EXPERIMENT_ID}.parquet"
FEATURE_COLUMNS_SOURCE_PATH = SOURCE_ML_FEATURE_DIR / "ml_feature_columns.csv"


# ------------------------------------------------------------
# 1.3 Export output paths (항상 로컬 BASE_DIR 기준)
# ------------------------------------------------------------
EXPORT_OUTPUT_DIR = (
    BASE_DIR
    / "ml"
    / "ml-00_baseline"
    / "outputs"
    / "feature_build"
    / EXPORT_NAME
    / EXPORT_EXPERIMENT_ID
    / RUN_ID
)

# split_single_parquet_by_existing_split(..., experiment_id=ARTIFACT_PREFIX)에 전달할 prefix
ARTIFACT_PREFIX = f"{EXPORT_NAME}__{EXPORT_EXPERIMENT_ID}__{RUN_ID}"

# split export 결과 파일 경로
TRAIN_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_train.parquet"
VAL_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_val.parquet"
TEST_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_test.parquet"

# split-only 실행 요약 파일 경로
SPLIT_SUMMARY_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_split_summary.csv"
SPLIT_BUILD_SUMMARY_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_split_existing_summary.json"

# ML 학습에 실제 사용할 ml_feature_columns.csv 사본 경로
FEATURE_COLUMNS_EXPORT_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_ml_feature_columns.csv"

# 이 노트북 자체의 export manifest 경로
MANIFEST_PATH = EXPORT_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_ml_inputs_manifest.json"


# ------------------------------------------------------------
# 1.4 Execution switches
# ------------------------------------------------------------
RUN_SPLIT_EXPORT = True
COPY_FEATURE_COLUMNS = True
SAVE_MANIFEST = True


# ------------------------------------------------------------
# 1.5 Overwrite policy
# ------------------------------------------------------------
# 기존 산출물이 있으면 실행을 중단한다. 재실행 시 RUN_ID 변경 권장.
OVERWRITE_POLICY = {
    "split_export": False,
    "feature_columns_copy": False,
    "manifest": False,
}


# ------------------------------------------------------------
# 1.6 Configuration preview
# ------------------------------------------------------------
print("=" * 80)
print("SOURCE_EXPERIMENT_ID        :", SOURCE_EXPERIMENT_ID)
print("EXPORT_EXPERIMENT_ID        :", EXPORT_EXPERIMENT_ID)
print("EXPORT_NAME                 :", EXPORT_NAME)
print("RUN_ID                      :", RUN_ID)
print("SOURCE_ML_FEATURE_DIR       :", SOURCE_ML_FEATURE_DIR)
print("INPUT_SINGLE_PARQUET        :", INPUT_SINGLE_PARQUET)
print("FEATURE_COLUMNS_SOURCE_PATH :", FEATURE_COLUMNS_SOURCE_PATH)
print("EXPORT_OUTPUT_DIR           :", EXPORT_OUTPUT_DIR)
print("TRAIN_PATH                  :", TRAIN_PATH)
print("VAL_PATH                    :", VAL_PATH)
print("TEST_PATH                   :", TEST_PATH)
print("SPLIT_SUMMARY_PATH          :", SPLIT_SUMMARY_PATH)
print("SPLIT_BUILD_SUMMARY_PATH    :", SPLIT_BUILD_SUMMARY_PATH)
print("FEATURE_COLUMNS_EXPORT_PATH :", FEATURE_COLUMNS_EXPORT_PATH)
print("MANIFEST_PATH               :", MANIFEST_PATH)
print("=" * 80)

## 2. 입력 계약 검증 함수 정의

단일 parquet와 ml_feature_columns.csv가 후속 ML 입력 계약을 만족하는지 확인하는 함수를 정의합니다. 실제 검증은 export 사본을 만든 뒤 `FEATURE_COLUMNS_EXPORT_PATH` 기준으로 수행합니다.

In [ ]:
def require_file(path: Path, label: str) -> None:
    """필수 입력 파일이 실제로 존재하는지 확인한다."""
    if not path.exists():
        raise FileNotFoundError(
            f"{label} not found: {path}\n"
            "전처리 산출물이 생성/복사되었는지, 또는 설정 셀의 경로가 맞는지 확인하세요."
        )
        
def require_no_existing_file(path: Path, *, overwrite: bool, label: str) -> None:
    """
    노트북에서 생성하는 보조 산출물의 overwrite를 차단
    split parquet 3개는 fb_build.split_single_parquet_by_existing_split() 내부에서 overwrite 정책을 검사
    이 함수는 모듈이 직접 관리하지 않는 보조 파일에만 사용한다. 예: feature_columns_path 복사, manifest 저장
    """
    if path.exists() and not overwrite:
        raise FileExistsError(
            f"Existing {label} found: {path}. "
            "Set overwrite=True or change RUN_ID to create a new output directory."
        )

def preview_input_contract(
    *,
    input_parquet_path: Path,
    feature_columns_path: Path,
    label_col: str,
) -> list[str]:
    """
    입력 데이터 .parquet 과 ml_feature_columns.csv가 ML 입력 export 계약을 만족하는지 확인한다.
    
    사용 목적
    --------
    - split export 전후에 parquet와 feature 선택 CSV의 정합성을 빠르게 확인한다.
    - 후속 ml_train.py / ml_val.py / ml_test.py가 읽을 수 있는 feature 목록을 확정한다.
    - 실제 feature column 파싱과 used_in_ml="TRUE"/"FALSE" strict 검증은  ml_io.check_feature_columns_file()에 위임한다.
      
    확인 항목
    --------
    - input parquet 파일 존재 여부
    - ml_feature_columns.csv 파일 존재 여부
    - parquet에 필수 meta 컬럼(tx_id, timestamp, split, label_col)이 있는지
    - used_in_ml="TRUE"로 선택된 feature 컬럼들이 parquet schema에 모두 있는지
    - 선택 feature 개수와 feature hash
    
    반환
    ----
    - used_in_ml="TRUE"인 feature column 목록
    """
    
    # 1. 입력 파일 존재 여부를 먼저 확인한다.
    require_file(input_parquet_path, "INPUT_SINGLE_PARQUET")
    require_file(feature_columns_path, "feature_columns_path")
    
    # 2. parquet 전체를 로드하지 않고 schema metadata에서 컬럼명만 확인한다.
    parquet_column_names = parquet_columns(input_parquet_path)
    parquet_column_set = set(parquet_column_names)
    
    # 3. 후속 ML 모듈이 공통으로 기대하는 최소 meta 컬럼을 확인한다.
    # tx_id: 거래 식별자
    # timestamp: 시간순 split 및 추적용 시각
    # split: train/val/test 역할 확인
    # label_col: 이진 분류 target
    required_meta_columns = {"tx_id", "timestamp", "split", label_col}
    missing_meta_columns = sorted(required_meta_columns - parquet_column_set)
    if missing_meta_columns:
        raise ValueError(f"single parquet is missing required meta columns: {missing_meta_columns}")
    
    # 4. ml_feature_columns.csv 검증은 ml_io 모듈에 위임한다.
    # 여기서는 used_in_ml="TRUE"/"FALSE" strict 정책, column_name 결측/중복, label/target 계열 누수 위험 feature 이름 차단을 수행한다.
    feature_check = check_feature_columns_file(
        feature_columns_path,
        project_root=PROJECT_ROOT,
        label_col=label_col,
        strict=True,
    )
    feature_columns = feature_check.selected_columns
    
    # 5. CSV에서 선택된 feature가 실제 parquet schema에 모두 존재하는지 확인한다.
    # 이 검증은 CSV의 column_name과 parquet 컬럼명의 최종 연결 지점이다.
    missing_features = [column for column in feature_columns if column not in parquet_column_set]
    if missing_features:
        raise ValueError(
            'used_in_ml="TRUE" feature columns are missing from input parquet. '
            f"missing={missing_features[:30]}, missing_count={len(missing_features)}"
        )
        
    # 6. 사람이 노트북에서 입력 계약 상태를 바로 확인할 수 있도록 핵심 요약을 출력한다.
    # feature_columns_hash는 feature 이름과 순서를 함께 반영한다.
    print("input_parquet_path        :", input_parquet_path)
    print("feature_columns_path      :", feature_columns_path)
    print("input parquet column_count:", len(parquet_column_names))
    print("selected feature_count    :", feature_check.selected_count)
    print("feature_columns_hash      :", feature_columns_hash(feature_columns))
    print("first_features            :", feature_columns[:10])
    print("input contract            : OK")
    return feature_columns
print("input contract helper functions loaded")

## 3. 원본 단일 parquet split 분포 확인

`split`, `label` 두 컬럼만 읽어서 원본 단일 parquet가 train/val/test를 모두 포함하는지 확인합니다.

In [ ]:
def summarize_split_labels(
    path: Path,
    *,
    expected_split: str | None = None,
    label_col: str = LABEL_COL,
) -> pd.DataFrame:
    """
    parquet의 split별 row 수와 label 분포를 확인한다.
    사용 목적
    --------
    - split export 전 원본 단일 parquet가 train/val/test를 모두 포함하는지 확인한다.
    - split export 후 개별 train/val/test parquet가 자기 split만 포함하는지 확인한다.
    - 후속 ML 학습 전에 각 split의 positive/negative 분포와 양 클래스 존재 여부를 빠르게 확인한다.
    주의
    ----
    - split은 train/val/test만 허용한다.
    - label은 0/1 이진값만 허용한다.
    - row 전체가 아니라 split, label_col 두 컬럼만 읽는다.
    """
    #  split과 label만 읽는다. 
    df = pd.read_parquet(path, columns=["split", label_col])
    
    # split 결측이 있으면 즉시 중단한다.
    if df["split"].isna().any():
        raise ValueError(f"split contains missing values. path={path}")
    
    # split 표기는 대소문자와 앞뒤 공백만 정규화한다. 예: " Train " -> "train"
    df["split"] = df["split"].astype("string").str.strip().str.lower()
    
    # 공백 문자열은 유효한 split 값이 아니므로 설정 오류로 처리한다.
    if (df["split"] == "").any():
        raise ValueError(f"split contains blank values. path={path}")
    
    # label은 숫자로 해석 가능한 0/1만 허용한다.. errors="raise"라서 숫자로 변환할 수 없는 값은 여기서 실패한다.
    labels = pd.to_numeric(df[label_col], errors="raise")
    
    # 0/1 외 label이 있으면 positive/negative count가 왜곡되므로 차단한다.
    observed_labels = set(labels.dropna().unique().tolist())
    if labels.isna().any() or not observed_labels <= {0, 1}:
        raise ValueError(f"label must be binary 0/1. path={path}, observed={sorted(observed_labels)}")
    
    # split export 된 파일에 대한 검증 필요시 사용
    # expected_split이 있으면 train/val/test 중 하나만 들어 있는 파일인지 확인한다.
    # expected_split이 없으면 원본 단일 parquet처럼 train/val/test가 모두 있어야 한다.
    expected = None if expected_split is None else expected_split.strip().lower()
    expected_values = {expected} if expected is not None else {"train", "val", "test"}
    observed = set(df["split"].unique().tolist())
    
    # 파일 내부 split 값이 기대한 split 구성과 정확히 일치하는지 확인한다.
    # 예: train parquet에 val/test row가 섞이면 여기서 실패한다.
    if observed != expected_values:
        raise ValueError(
            f"unexpected split values in {path}: "
            f"expected={sorted(expected_values)}, observed={sorted(observed)}"
        )
    # 출력 순서를 고정한다. set 기반 observed 순서를 쓰면 표시 순서가 불안정해진다.
    split_order = [expected] if expected is not None else ["train", "val", "test"]
    rows = []
    for split_name in split_order:
        # 현재 split에 해당하는 row만 선택한다.
        part = df[df["split"] == split_name]
        # labels는 df와 같은 index를 유지하므로 part.index로 같은 row의 label만 가져온다.
        part_labels = labels.loc[part.index]
        # label=1을 positive class로 본다.
        positive_count = int((part_labels == 1).sum())
        rows.append({
            "split": split_name,
            "rows": int(len(part)),
            "positive_count": positive_count,
            "negative_count": int((part_labels == 0).sum()),
            "positive_rate": float(positive_count / len(part)),
            "both_classes": 0 < positive_count < len(part),
        })
    return pd.DataFrame(rows)


input_split_summary = summarize_split_labels(
    INPUT_SINGLE_PARQUET,
    label_col=LABEL_COL,
)
display(input_split_summary)

## 4. 단일 parquet -> ML 입력 parquet 생성

`fb_build.split_single_parquet_by_existing_split()`만 사용합니다. 이 함수는 feature를 새로 계산하지 않고 기존 `split` 컬럼을 기준으로 파일을 나눕니다.

In [ ]:
def export_split_parquets():
    """
    전처리 단일 parquet를 후속 ML 모듈 입력용 split parquet 3개로 분리한다.
    이 함수는 feature를 새로 만들지 않는다.
    입력 parquet에 이미 존재하는 split 컬럼을 기준으로 train/val/test 파일만 export한다.
    """
    if not RUN_SPLIT_EXPORT:
        print("Split export skipped. Existing split files will be validated if present.")
        return None
    result = split_single_parquet_by_existing_split(
        input_path=INPUT_SINGLE_PARQUET,
        output_dir=EXPORT_OUTPUT_DIR,
        base_dir=PROJECT_ROOT,
        experiment_id=ARTIFACT_PREFIX,
        overwrite=OVERWRITE_POLICY["split_export"],
        tx_id_col="tx_id",
        timestamp_col="timestamp",
        label_col=LABEL_COL,
        split_col="split",
    )
    print("row_counts:", result.row_counts)
    print("train_path:", result.output_paths.train_path)
    print("val_path  :", result.output_paths.val_path)
    print("test_path :", result.output_paths.test_path)
    display(result.split_summary)
    return result


split_export_result = export_split_parquets()

In [ ]:
def prepare_feature_columns_for_ml() -> Path:
    """
    ML 학습에 사용할 ml_feature_columns.csv 사본을 준비한다.
    사용 목적
    --------
    - 원본 FEATURE_COLUMNS_SOURCE_PATH는 전처리 산출물이므로 직접 수정하지 않는다.
    - ML 학습에는 output directory에 복사된 FEATURE_COLUMNS_EXPORT_PATH 사본을 사용한다.
    - export 사본의 used_in_ml 표기를 "TRUE" / "FALSE" 문자열 정책으로 정규화한다.
    동작 방식
    --------
    - COPY_FEATURE_COLUMNS=True:
      원본 ml_feature_columns.csv를 export output directory로 복사한다.
    - COPY_FEATURE_COLUMNS=False:
      사용자가 이미 준비한 FEATURE_COLUMNS_EXPORT_PATH를 그대로 사용한다.
    - 두 경우 모두 마지막에 normalize_feature_columns_file()을 호출해
      used_in_ml 표기를 "TRUE" / "FALSE"로 통일한다.
    """
    if COPY_FEATURE_COLUMNS:
        # 원본 feature columns CSV가 실제로 존재하는지 먼저 확인한다.
        require_file(FEATURE_COLUMNS_SOURCE_PATH, "FEATURE_COLUMNS_SOURCE_PATH")
        
        # export 사본을 조용히 덮어쓰지 않도록 보호한다.
        require_no_existing_file(
            FEATURE_COLUMNS_EXPORT_PATH,
            overwrite=OVERWRITE_POLICY["feature_columns_copy"],
            label="feature columns export",
        )
        
        # output directory가 없으면 생성한 뒤 원본 CSV를 사본 경로로 복사한다.
        FEATURE_COLUMNS_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(FEATURE_COLUMNS_SOURCE_PATH, FEATURE_COLUMNS_EXPORT_PATH)
        print("feature_columns copied:", FEATURE_COLUMNS_EXPORT_PATH)
    else:
        # COPY_FEATURE_COLUMNS=False이면 사용자가 이미 준비한 사본을 검증 대상으로 사용한다.
        require_file(FEATURE_COLUMNS_EXPORT_PATH, "FEATURE_COLUMNS_EXPORT_PATH")
        print("feature_columns copy skipped. Using existing export CSV:", FEATURE_COLUMNS_EXPORT_PATH)
        
    # 기존 산출물에는 True/False bool 표기가 남아 있을 수 있다.
    # 학습 입력 계약은 "TRUE" / "FALSE" 문자열이므로 export 사본을 정규화한다.
    normalized = normalize_feature_columns_file(
        FEATURE_COLUMNS_EXPORT_PATH,
        project_root=PROJECT_ROOT,
        label_col=LABEL_COL,
        overwrite=True,
        strict=True,
    )
    print("feature_columns normalized: used_in_ml uses TRUE/FALSE only")
    print("selected feature_count after normalization:", normalized.selected_count)
    
    # 후속 preview_input_contract(), ml_train.py 등에 넘길 최종 CSV 경로를 반환한다.
    return FEATURE_COLUMNS_EXPORT_PATH

prepare_feature_columns_for_ml()
# ML 학습에 들어갈 최종 ml_feature_columns.csv는 FEATURE_COLUMNS_EXPORT_PATH다.
# 이 파일 기준으로 used_in_ml="TRUE" feature 목록과 parquet schema 정합성을 검증한다.
FEATURE_COLUMNS = preview_input_contract(
    input_parquet_path=INPUT_SINGLE_PARQUET,
    feature_columns_path=FEATURE_COLUMNS_EXPORT_PATH,
    label_col=LABEL_COL,
)

## 5. 생성 산출물 검증

생성된 split parquet 3개가 후속 ML 모듈 입력 계약을 만족하는지 확인합니다.

In [ ]:
def validate_exported_split_file(
    path: Path,
    *,
    expected_split: str,
    feature_columns: list[str],
) -> pd.DataFrame:
    """
    후속 ml_*.py 입력으로 사용할 split parquet 하나를 검증한다.
    
    확인 항목
    --------
    - parquet 파일 존재 여부
    - parquet schema에 label, split, selected feature columns가 모두 있는지
    - 파일 내부 split 값이 expected_split 하나로만 구성되어 있는지
    - label 분포와 양 클래스 존재 여부
    
    주의
    ----
    - feature_columns는 ml_feature_columns.csv에서 used_in_ml="TRUE"로 선택된 컬럼 목록이다.
    - 이 함수는 parquet row 전체를 읽기 전에 schema metadata로 컬럼 존재 여부를 먼저 확인한다.
    """
    
    # export된 train/val/test parquet 파일이 실제로 존재하는지 먼저 확인한다.
    require_file(path, f"{expected_split}_parquet")
    
    # parquet 전체를 로드하지 않고 schema metadata에서 컬럼명만 확인한다.
    schema_names = set(parquet_columns(path))
    
    # 후속 ml_train.py / ml_val.py / ml_test.py가 읽기 위해 필요한 최소 컬럼이다.
    # feature_columns: 모델 입력 X
    # LABEL_COL: 정답 y
    # split: train/val/test 역할 검증용
    required_columns = set(feature_columns) | {LABEL_COL, "split"}
    
    # 선택 feature 또는 필수 meta 컬럼이 누락되면 후속 ML 단계에서 실패하므로 여기서 먼저 차단한다.
    missing = sorted(required_columns - schema_names)
    if missing:
        raise ValueError(
            f"Exported {expected_split} parquet is missing required columns. "
            f"missing={missing[:30]}, missing_count={len(missing)}, path={path}"
        )
        
    # split 값과 label 분포를 확인한다.
    # expected_split="train"이면 파일 내부 split 값도 train만 있어야 한다.
    return summarize_split_labels(path, expected_split=expected_split)


export_summaries = []

# export된 train/val/test 파일을 모두 같은 기준으로 검증한다.
for split_name, split_path in {"train": TRAIN_PATH, "val": VAL_PATH, "test": TEST_PATH}.items():
    summary = validate_exported_split_file(
        split_path,
        expected_split=split_name,
        feature_columns=FEATURE_COLUMNS,
    )
    # 여러 split 요약을 합쳤을 때 어느 파일에서 나온 결과인지 추적하기 위해 path 컬럼을 추가한다.
    summary.insert(0, "path", str(split_path))
    export_summaries.append(summary)
    
# split별 검증 결과를 하나의 DataFrame으로 합쳐 노트북에서 바로 확인한다.
export_split_summary = pd.concat(export_summaries, ignore_index=True)
display(export_split_summary)
print("export validation: OK")

## 6. Manifest 저장

후속 `ml_train.py`, `ml_val.py`, `ml_test.py`에 넘길 경로와 feature fingerprint를 JSON으로 저장합니다.

In [ ]:
def save_manifest() -> dict:
    """
    후속 ML 실행에 필요한 입력 경로와 검증 요약을 manifest JSON으로 저장한다.
    사용 목적
    --------
    - 이 노트북이 어떤 원본 parquet와 어떤 ml_feature_columns.csv 사본을 사용했는지 기록한다.
    - 후속 ml_train.py / ml_val.py / ml_test.py에 넘길 입력 경로를 한곳에 남긴다.
    - feature 목록과 feature hash를 저장해 학습 시점 feature 순서 재현성을 확인할 수 있게 한다.
    - split별 label 분포 요약을 함께 저장해 train/val/test 입력 검증 결과를 추적한다.
    
    주의
    ----
    - 이 manifest는 학습 결과가 아니다.
    - 이 노트북은 XGBoost 학습, validation threshold 선택, final test를 실행하지 않는다.
    - 기존 manifest가 있으면 overwrite 정책에 따라 저장을 차단한다.
    """
    # 후속 실행과 재현성 확인에 필요한 경로, feature 목록, split 요약을 하나의 dict로 모은다.
    manifest = {

        "notebook": "ml/ml-00_baseline/feature_build/fb_ml00_baseline_test_run.ipynb",         # 어떤 노트북이 이 manifest를 만들었는지 기록한다.
        "purpose": "Export ML-00 input artifacts from 03_ml_feature_process single parquet.",  # manifest의 목적을 명시
        "manifest_type": "ml_input_export",
        
        # 실행 식별 및 재현성 metadata.
        "seed": SEED,
        "source_experiment_id": SOURCE_EXPERIMENT_ID,
        "export_experiment_id": EXPORT_EXPERIMENT_ID,
        "export_name": EXPORT_NAME,
        "run_id": RUN_ID,
        "artifact_prefix": ARTIFACT_PREFIX,
        
        # 입력 원본과 ML feature 선택 기준 파일의 provenance.
        "input_single_parquet": str(INPUT_SINGLE_PARQUET),
        "feature_columns_source": str(FEATURE_COLUMNS_SOURCE_PATH),
        "feature_columns_for_ml": str(FEATURE_COLUMNS_EXPORT_PATH),
        
        # 후속 ml_train.py / ml_val.py / ml_test.py에 넘길 주요 산출물 경로.
        "outputs": {
            "train_path": str(TRAIN_PATH),
            "val_path": str(VAL_PATH),
            "test_path": str(TEST_PATH),
            "feature_columns_path": str(FEATURE_COLUMNS_EXPORT_PATH),
            "split_summary_path": str(SPLIT_SUMMARY_PATH),
            "split_existing_summary_path": str(SPLIT_BUILD_SUMMARY_PATH),
        },
        
        # 모델 입력 feature 목록과 순서 fingerprint.
        # feature_columns_hash는 feature 이름뿐 아니라 순서까지 반영한다.
        "feature_count": len(FEATURE_COLUMNS),
        "feature_columns_hash": feature_columns_hash(FEATURE_COLUMNS),
        "feature_columns": FEATURE_COLUMNS,
        
        # 원본 단일 parquet와 export된 split parquet의 label 분포 검증 요약.
        "input_split_summary": input_split_summary.to_dict(orient="records"),
        "export_split_summary": export_split_summary.to_dict(orient="records"),
    }
    
    # SAVE_MANIFEST=False이면 파일 저장 없이 manifest dict만 반환한다.
    if SAVE_MANIFEST:
        # 기존 manifest를 조용히 덮어쓰지 않도록 overwrite 정책을 적용한다.
        require_no_existing_file(
            MANIFEST_PATH,
            overwrite=OVERWRITE_POLICY["manifest"],
            label="manifest",
        )
        
        # manifest 저장 경로의 parent directory가 없으면 생성한다.
        MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
        
        # 사람이 diff/review하기 쉽도록 indent=2 JSON으로 저장한다.
        MANIFEST_PATH.write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        print("manifest saved:", MANIFEST_PATH)
    return manifest
ML_INPUT_MANIFEST = save_manifest()

## 7. 후속 ML 모듈 입력 경로

아래 경로를 후속 `ml_train.py`, `ml_val.py`, `ml_test.py` 실행 설정에 넣으면 됩니다. 이 노트북은 여기서 종료합니다.

In [ ]:
artifact_paths = {
    "train_path": TRAIN_PATH,
    "val_path": VAL_PATH,
    "test_path": TEST_PATH,
    "split_summary_path": SPLIT_SUMMARY_PATH,
    "split_build_summary_path": SPLIT_BUILD_SUMMARY_PATH,
    "feature_columns_path": FEATURE_COLUMNS_EXPORT_PATH,
    "manifest_path": MANIFEST_PATH,
}
artifact_status = pd.DataFrame(
    [{"name": name, "exists": path.exists(), "path": str(path)} for name, path in artifact_paths.items()]
)
display(artifact_status)

print("후속 ml_train.py 입력 예시")
print("train_path           =", TRAIN_PATH)
print("val_path             =", VAL_PATH)
print("feature_columns_path =", FEATURE_COLUMNS_EXPORT_PATH)
print()
print("후속 ml_val.py 입력 예시")
print("val_path             =", VAL_PATH)
print()
print("후속 ml_test.py 입력 예시")
print("test_path            =", TEST_PATH)
print()
print("추적용 export 산출물")
print("split_summary_path       =", SPLIT_SUMMARY_PATH)
print("split_build_summary_path =", SPLIT_BUILD_SUMMARY_PATH)
print("manifest_path            =", MANIFEST_PATH)
print()
print("주의: 이 노트북은 학습/validation/test를 실행하지 않았습니다.")